In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from os import environ

import boto3

profile_name = "odin-cdk"
session = boto3.Session(profile_name=profile_name)
credentials = session.get_credentials()
if credentials:
    environ["AWS_SECRET_ACCESS_KEY"] = credentials.secret_key
    environ["AWS_ACCESS_KEY_ID"] = credentials.access_key

In [3]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [4]:
from handler.create_zpt_handler import get_scan_data, merge_era5, read_era5

from tests.conftest import scan_data
import xarray as xr


ds_scans = get_scan_data(scan_data["ScansInfo"])
print(ds_scans)
ds_era5 = read_era5(ds_scans)
print(ds_era5)
ds = merge_era5(ds_scans, ds_era5)

<xarray.Dataset> Size: 6kB
Dimensions:   (ScanID: 53)
Coordinates:
  * ScanID    (ScanID) int64 424B 14884154927 14884157198 ... 14884274601
Data variables: (12/14)
    AltStart  (ScanID) float64 424B 5.683e+03 1.036e+05 ... 5.736e+03 1.053e+05
    AltEnd    (ScanID) float64 424B 1.039e+05 6.576e+03 ... 1.036e+05 8.588e+03
    LatStart  (ScanID) float64 424B 37.18 30.62 19.47 ... -58.91 -69.73 -75.74
    LatEnd    (ScanID) float64 424B 30.76 19.61 13.08 ... -69.58 -75.56 -82.48
    LonStart  (ScanID) float64 424B 95.75 93.91 91.55 ... 47.63 38.89 28.13
    LonEnd    (ScanID) float64 424B 93.95 91.58 90.07 88.01 ... 39.06 28.6 337.8
    ...        ...
    FreqMode  (ScanID) int64 424B 19 19 19 19 19 19 19 ... 19 19 19 19 19 19 19
    Backend   (ScanID) object 424B 'AC1' 'AC1' 'AC1' 'AC1' ... 'AC1' 'AC1' 'AC1'
    MJDMid    (ScanID) float64 424B 6.069e+04 6.069e+04 ... 6.069e+04 6.069e+04
    LatMid    (ScanID) float64 424B 33.97 25.12 16.28 ... -64.31 -72.71 -80.02
    LonMid    (ScanID

/home/joakim/.virtual_envs/create_zpt/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


<xarray.Dataset> Size: 137MB
Dimensions:    (time: 4, level: 37, latitude: 241, longitude: 480)
Coordinates:
  * latitude   (latitude) float64 2kB 90.0 89.25 88.5 ... -88.5 -89.25 -90.0
  * level      (level) float64 296B 1e+03 975.0 950.0 925.0 ... 5.0 3.0 2.0 1.0
  * longitude  (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
  * time       (time) datetime64[ns] 32B 2025-01-12 ... 2025-01-12T18:00:00
Data variables:
    t          (time, level, latitude, longitude) float32 68MB dask.array<chunksize=(1, 10, 121, 240), meta=np.ndarray>
    z          (time, level, latitude, longitude) float32 68MB dask.array<chunksize=(1, 10, 121, 240), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-01-22T23:43 GRIB to

In [5]:
xr.set_options(display_style="text")
ds

<xarray.Dataset> Size: 54kB
Dimensions:     (ScanID: 53, level: 37)
Coordinates:
  * ScanID      (ScanID) int64 424B 14884154927 14884157198 ... 14884274601
  * level       (level) float64 296B 1e+03 975.0 950.0 925.0 ... 5.0 3.0 2.0 1.0
Data variables: (12/19)
    AltStart    (ScanID) float64 424B 5.683e+03 1.036e+05 ... 1.053e+05
    AltEnd      (ScanID) float64 424B 1.039e+05 6.576e+03 ... 8.588e+03
    LatStart    (ScanID) float64 424B 37.18 30.62 19.47 ... -58.91 -69.73 -75.74
    LatEnd      (ScanID) float64 424B 30.76 19.61 13.08 ... -69.58 -75.56 -82.48
    LonStart    (ScanID) float64 424B 95.75 93.91 91.55 ... 47.63 38.89 28.13
    LonEnd      (ScanID) float64 424B 93.95 91.58 90.07 ... 39.06 28.6 337.8
    ...          ...
    DateMid     (ScanID) datetime64[ns] 424B 2025-01-12T00:47:59.413919 ... 2...
    era5_level  (level) float64 296B 1e+03 975.0 950.0 925.0 ... 5.0 3.0 2.0 1.0
    era5_z      (ScanID, level) float32 8kB dask.array<chunksize=(53, 10), meta=np.ndarray>
    era5_t      (ScanID, level) float32 8kB dask.array<chunksize=(53, 10), meta=np.ndarray>
    era5_gmh    (ScanID, level) float64 16kB dask.array<chunksize=(53, 10), meta=np.ndarray>
    theta       (ScanID, level) float64 16kB dask.array<chunksize=(53, 10), meta=np.ndarray>

In [6]:
from handler.newdonalettyERANC import Donaletty
import numpy as np

ecmz = np.arange(45)
newz = np.arange(151)
donaletty = Donaletty()
scan_on_interp_gmh = ds.groupby("ScanID").map(
    func=donaletty.interpolate_gmh, args=(ecmz,)
)

In [7]:
scan_on_interp_gmh

<xarray.Dataset> Size: 83kB
Dimensions:     (ScanID: 53, era5_gmh: 45)
Coordinates:
    level       (ScanID, era5_gmh) float64 19kB 273.0 898.1 ... 2.652 2.362
  * era5_gmh    (era5_gmh) int64 360B 0 1 2 3 4 5 6 7 ... 38 39 40 41 42 43 44
  * ScanID      (ScanID) int64 424B 14884154927 14884157198 ... 14884274601
Data variables: (12/18)
    AltStart    (ScanID) float64 424B 5.683e+03 1.036e+05 ... 1.053e+05
    AltEnd      (ScanID) float64 424B 1.039e+05 6.576e+03 ... 8.588e+03
    LatStart    (ScanID) float64 424B 37.18 30.62 19.47 ... -58.91 -69.73 -75.74
    LatEnd      (ScanID) float64 424B 30.76 19.61 13.08 ... -69.58 -75.56 -82.48
    LonStart    (ScanID) float64 424B 95.75 93.91 91.55 ... 47.63 38.89 28.13
    LonEnd      (ScanID) float64 424B 93.95 91.58 90.07 ... 39.06 28.6 337.8
    ...          ...
    LonMid      (ScanID) float64 424B 94.81 92.69 90.8 ... 44.18 34.58 11.15
    DateMid     (ScanID) datetime64[ns] 424B 2025-01-12T00:47:59.413919 ... 2...
    era5_level  (ScanID, era5_gmh) float64 19kB 273.0 898.1 ... 2.652 2.362
    era5_z      (ScanID, era5_gmh) float32 10kB dask.array<chunksize=(1, 45), meta=np.ndarray>
    era5_t      (ScanID, era5_gmh) float32 10kB dask.array<chunksize=(1, 45), meta=np.ndarray>
    theta       (ScanID, era5_gmh) float64 19kB dask.array<chunksize=(1, 45), meta=np.ndarray>

In [19]:
zpt_donaletty = scan_on_interp_gmh.groupby("ScanID").map(
    donaletty.donaletty, args=(newz,)
)

/home/joakim/.virtual_envs/create_zpt/lib/python3.11/site-packages/dask/array/core.py:1729: FutureWarning: The `numpy.interp` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(
/home/joakim/.virtual_envs/create_zpt/lib/python3.11/site-packages/dask/array/core.py:1729: FutureWarning: The `numpy.interp` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(
/home/joakim/.virtual_envs/create_zpt/lib/python3.11/site-packages/dask/array/core.py:1729: FutureWarning: The `numpy.interp` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(
/home/joakim/.virtu

In [20]:
zpt_donaletty

<xarray.Dataset> Size: 130kB
Dimensions:  (ScanID: 53, z: 151)
Coordinates:
  * z        (z) int64 1kB 0 1 2 3 4 5 6 7 8 ... 143 144 145 146 147 148 149 150
  * ScanID   (ScanID) int64 424B 14884154927 14884157198 ... 14884274601
Data variables:
    p        (ScanID, z) float64 64kB 1.054e+03 927.9 ... 6.212e-06 6.004e-06
    t        (ScanID, z) float64 64kB 273.0 267.9 274.7 ... 784.6 794.7 804.6